# 🧮 Clase 02 · Inversa, determinante y regresión lineal desde cero

**Álgebra Lineal · UMAT205 · UEES · Semana 2**

Hoy programamos exactamente lo que vimos en la pizarra: calcular la inversa de una matriz,
usar el determinante como criterio de invertibilidad, y construir desde cero los pesos de una
regresión lineal $w=(X^\top X)^{-1}X^\top y$ — incluyendo saber diagnosticar cuándo el modelo
no se puede ajustar por multicolinealidad.

**Reto oficial de la semana ("Regresión lineal desde cero"):** se califica con —
(1) cálculo correcto de $X^\top X$, su determinante e inversa (40%),
(2) obtención de los pesos $w=(X^\top X)^{-1}X^\top y$ (30%),
(3) diagnóstico e interpretación de la multicolinealidad ante matrices singulares (30%).


In [ ]:
import numpy as np


## 1. La inversa con NumPy

`np.linalg.det(A)` calcula el determinante y `np.linalg.inv(A)` calcula la inversa directamente
— son el equivalente en código de la fórmula directa (2×2) y de Gauss-Jordan sobre $[A\mid I]$
(para cualquier tamaño) que viste en la pizarra.


In [ ]:
A = np.array([[3, 1],
              [2, 1]])

det = np.linalg.det(A)
A_inv = np.linalg.inv(A)

print("det(A) =", det)
print("A_inv =\n", A_inv)
print("Verificación A @ A_inv =\n", np.round(A @ A_inv, 6))   # debe dar la identidad


### 🎯 Reto 1
Con $A=\begin{pmatrix}4&3\\2&2\end{pmatrix}$:
1. Calcula `det` y `A_inv` con NumPy.
2. Verifica que `A @ A_inv` da (aproximadamente) la identidad.
3. **Bonus:** compara `A_inv` con lo que te da la fórmula directa a mano — deben coincidir.


In [ ]:
# Tu código aquí 👇


## 2. Cuando la matriz es singular

Si $\det(A)=0$, `np.linalg.inv(A)` **no devuelve un resultado incorrecto silenciosamente** —
lanza un error (`LinAlgError: Singular matrix`). Es buena práctica revisar el determinante ANTES
de intentar invertir, sobre todo en el reto de regresión de abajo.


In [ ]:
A_singular = np.array([[2, 4],
                        [1, 2]])   # fila 2 = 0.5 * fila 1 -> columnas/filas proporcionales

print("det(A_singular) =", np.linalg.det(A_singular))

try:
    np.linalg.inv(A_singular)
except np.linalg.LinAlgError as e:
    print("⚠️ No se pudo invertir:", e)


## 3. La matriz de diseño X y la regresión lineal desde cero

El mismo ejemplo de la pizarra: 3 estudiantes, horas de estudio ($x$) y su nota ($y$).
Construimos $X$ con una columna de 1's (intercepto) + la variable, y aplicamos la ecuación
normal $w=(X^\top X)^{-1}X^\top y$.


In [ ]:
X = np.array([[1, 1],
              [1, 2],
              [1, 3]])
y = np.array([52, 58, 65])

XtX = X.T @ X
det_XtX = np.linalg.det(XtX)
print("X^T X =\n", XtX)
print("det(X^T X) =", det_XtX)

assert abs(det_XtX) > 1e-9, "X^T X es singular: revisa si hay columnas redundantes en X"

w = np.linalg.inv(XtX) @ X.T @ y
print("w = (X^T X)^-1 X^T y =", w)
print(f"\nInterpretación: cada hora extra de estudio suma {w[1]:.2f} puntos a la nota "
      f"(según estos {len(y)} datos). Intercepto (0 horas, extrapolación): {w[0]:.2f}.")


### 🎯 Reto 2
Otro dataset real: horas de sueño ($x$) y puntaje de concentración ($y$) de un estudiante:
$(x,y) = (4, 60),\ (6, 72),\ (8, 78)$.

1. Arma la matriz de diseño `X` (con la columna de 1's) y el vector `y`.
2. Calcula `X.T @ X` y su determinante — confirma que es invertible.
3. Obtén `w` con la ecuación normal e interpreta el resultado (¿cuánto suma cada hora extra de sueño?).


In [ ]:
# Tu código aquí 👇


## 4. Multicolinealidad: diagnóstico

¿Qué pasa si agregamos una columna que no aporta información nueva — por ejemplo, la misma
variable en otra escala? Repetimos el ejemplo de la Sección 3, pero agregando una tercera
columna redundante.


In [ ]:
X2 = np.array([[1, 1, 2],
               [1, 2, 4],
               [1, 3, 6]])   # columna 3 = 2 * columna 2 (redundante)
y2 = np.array([52, 58, 65])

XtX2 = X2.T @ X2
det2 = np.linalg.det(XtX2)
print("det(X2^T X2) =", det2)

if abs(det2) < 1e-6:
    print("⚠️ Multicolinealidad detectada: X^T X es (prácticamente) singular.")
    print("   Causa probable: dos columnas de X son proporcionales entre sí.")
else:
    w2 = np.linalg.inv(XtX2) @ X2.T @ y2
    print("w =", w2)


### 🎯 Reto 3
Toma el dataset del **Reto 2** (horas de sueño → concentración) y arma una nueva matriz de
diseño `X3` agregando una tercera columna redundante: el DOBLE de las horas de sueño.

1. Calcula `det(X3.T @ X3)` — debe salir (prácticamente) 0.
2. Con un `try/except` (como en la Sección 2), intenta invertir `X3.T @ X3` y captura el error.
3. Imprime una línea explicando, en tus palabras, cuál columna es la redundante y por qué.


In [ ]:
# Tu código aquí 👇


## 5. 🔗 Reto de enlace — antes de la próxima clase

La Clase 03 resuelve $Ax=b$ con el **método de Gauss** para modelar un mini-perceptrón — sin
pasar por la inversa (que es cara de calcular cuando el sistema es grande).

Antes de la próxima sesión, resuelve este sistema con CUALQUIER método que ya conozcas
(`np.linalg.solve`, o construyendo la inversa a mano):

$$\begin{cases}2x+y=8\\x-y=1\end{cases}$$

Trae tu respuesta a la próxima clase — ahí la comparamos con el método de Gauss directo.


In [ ]:
# Tu código aquí 👇


---
## ✅ Checklist de salida

- [ ] Calculo `det` y `A_inv` con NumPy y verifico `A @ A_inv ≈ I`.
- [ ] Sé por qué `np.linalg.inv` lanza `LinAlgError` cuando la matriz es singular.
- [ ] Construí la matriz de diseño `X` (con columna de 1's) para un problema de regresión.
- [ ] Obtuve los pesos `w = (X^T X)^-1 X^T y` desde cero, sin librerías de machine learning.
- [ ] Diagnostiqué multicolinealidad calculando `det(X^T X)` con una columna redundante.

**Tarea:** completa la actividad de la Unidad 1 (Semana 2) en Blackboard (plazo: 7 días).

[⬅ Volver al sitio del curso](https://pdavicho.github.io/algebra-lineal-uees/clases/clase-02/index.html)
